# 構文木を括弧表記で読み書きし、操作する (NLTK Tree)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/18_parsing/tree_bracket.ipynb)

NLP クックブック 18-16 ／ 構文解析

Penn Treebank 形式の括弧付き構文木を読み込んで、表示したり、部分木を取り出したり、文法規則を数えたりしたい。

In [1]:
from nltk import Tree

s = "(S (NP (DT The) (JJ quick) (NN fox)) (VP (VBD jumped) (PP (IN over) (NP (DT the) (JJ lazy) (NN dog)))) (. .))"
tree = Tree.fromstring(s)
tree.pretty_print()
print("単語:", tree.leaves())
print("品詞:", tree.pos())
print("高さ:", tree.height(), " ラベル:", tree.label())

                      S                       
       _______________|_____________________   
      |                    VP               | 
      |          __________|___             |  
      |         |              PP           | 
      |         |      ________|___         |  
      NP        |     |            NP       | 
  ____|____     |     |     _______|____    |  
 DT   JJ   NN  VBD    IN   DT      JJ   NN  . 
 |    |    |    |     |    |       |    |   |  
The quick fox jumped over the     lazy dog  . 

単語: ['The', 'quick', 'fox', 'jumped', 'over', 'the', 'lazy', 'dog', '.']
品詞: [('The', 'DT'), ('quick', 'JJ'), ('fox', 'NN'), ('jumped', 'VBD'), ('over', 'IN'), ('the', 'DT'), ('lazy', 'JJ'), ('dog', 'NN'), ('.', '.')]
高さ: 6  ラベル: S


In [2]:
# 部分木の取り出し
for sub in tree.subtrees(lambda t: t.label() == "NP"):
    print("NP:", " ".join(sub.leaves()))
print("PP の位置:", [pos for pos in tree.treepositions() if isinstance(tree[pos], Tree) and tree[pos].label() == "PP"])
print(tree[1, 1])                       # 位置を指定してアクセス (VP の 2 番目の子)

# 木から文法規則 (生成規則) を取り出す
for prod in tree.productions()[:6]:
    print(" ", prod)

NP: The quick fox
NP: the lazy dog
PP の位置: [(1, 1)]
(PP (IN over) (NP (DT the) (JJ lazy) (NN dog)))
  S -> NP VP .
  NP -> DT JJ NN
  DT -> 'The'
  JJ -> 'quick'
  NN -> 'fox'
  VP -> VBD PP


In [3]:
# ツリーバンクから PCFG を推定する
import nltk
nltk.download("treebank", quiet=True)
from nltk.corpus import treebank

productions = [p for t in treebank.parsed_sents()[:500] for p in t.productions()]
grammar = nltk.induce_pcfg(nltk.Nonterminal("S"), productions)
print(len(grammar.productions()), "規則")
for p in sorted(grammar.productions(lhs=nltk.Nonterminal("NP")), key=lambda p: -p.prob())[:5]:
    print(f"  {p.prob():.3f}  {p}")

5508 規則
  0.088  NP -> NP PP [0.0877309]
  0.079  NP -> DT NN [0.0791557]
  0.054  NP -> -NONE- [0.0537599]
  0.046  NP -> NN [0.0455145]
  0.044  NP -> NNP [0.0441953]


In [4]:
# 木の変形: チョムスキー標準形 (二分木) への変換と、括弧表記への書き戻し
t = Tree.fromstring("(NP (DT the) (JJ big) (JJ red) (NN ball))")
t.chomsky_normal_form()
print(t)
print(" ".join(str(tree).split()))      # 1 行の括弧表記に戻す

(NP
  (DT the)
  (NP|<JJ-JJ-NN> (JJ big) (NP|<JJ-NN> (JJ red) (NN ball))))
(S (NP (DT The) (JJ quick) (NN fox)) (VP (VBD jumped) (PP (IN over) (NP (DT the) (JJ lazy) (NN dog)))) (. .))


- 括弧表記は `(ラベル 子1 子2 ...)` の入れ子。句構造のツリーバンク（Penn Treebank、国立国語研究所の NPCMJ、Stanza の構文解析結果など）はほぼこの形式で配布されている。
- CKY アルゴリズムなど多くの構文解析アルゴリズムは二分木を前提にするので、学習前に二分化し、評価前に元に戻す。
- 句構造解析の評価は、ラベル付き括弧の適合率・再現率（PARSEVAL、`evalb`）で行う。